# Lab 9.1 — Capstone: One Ticket, End to End

*Chapter 9 — Course Summary · 50 minutes · JupyterLab + pytest + the OpenAI API via `course_ai`*

One small ticket, the whole GenAI workflow, no shortcuts. Draft acceptance
criteria with the model and **edit them like a reviewer**; generate the tests
FIRST and watch them fail; generate the implementation and iterate to green;
gate it with a tiny eval suite; generate the docs; then fill in a process
scorecard — what you verified, and what you took on trust.

The ticket is small on purpose: the workflow is the deliverable.

## Objectives

By the end of this lab, you will:

- Run the full sequence: criteria → tests-first (red) → implementation (green)
  → eval gate → docs → process scorecard.
- Reuse the course idioms where they belong: prompt patterns (Ch04), the AI-TDD
  loop (Lab 5.1), the eval gate (Lab 6.1).
- End with an explicit, written record of what you verified vs what you trusted.

![One ticket crosses every phase — the GenAI-augmented SDLC, with the human accountable column you keep](diagrams/ch01_sdlc_map.png)

*One ticket crosses every phase — the GenAI-augmented SDLC, with the human accountable column you keep (Chapter 1 deck).*

## Setup

- **Key:** `OPENAI_API_KEY` from the environment / course `.env`, never printed.
  `COURSE_AI_MOCK=1` (or no key) runs the whole workflow offline against the
  deterministic canned assets in `course_ai.CANNED_91` — every stage still
  produces real files, real test runs, and a real gate.
- **Workspace:** the notebook creates `capstone_workspace/` next to itself and
  writes the module and tests there. Regenerated every run; safe to delete.
- **Continuity:** nothing here is new machinery — that is the point. The course's
  moves, chained into one ticket.

In [ ]:
import importlib
import pathlib
import re
import subprocess
import sys

import course_ai
from course_ai import chat, chat_json

print("mode:", course_ai.mode())

WORKSPACE = pathlib.Path("capstone_workspace")
WORKSPACE.mkdir(exist_ok=True)

TEXT_UTILS_V0 = """# Small text utilities for the publishing pipeline.


def initials(name):
    return ".".join(p[0] for p in name.split()) + "."
"""

def run_tests():
    """Run pytest on the capstone workspace; return (green, output)."""
    r = subprocess.run([sys.executable, "-m", "pytest", str(WORKSPACE),
                        "-q", "--tb=line", "-rf"], capture_output=True, text=True, timeout=120)
    return r.returncode == 0, (r.stdout + r.stderr).strip()

def tail(output, n=10):
    """The last n lines — the part of a pytest run you actually read."""
    return "\n".join(output.splitlines()[-n:])

history = []   # (label, green) — the red -> green loop, captured

## Steps

### Step 1 — Acceptance criteria, drafted then EDITED (8 min)

The model drafts; you edit. The draft is a starting point, not a spec — read
every criterion against the ticket and fix what it missed or over-specified
*before anything else is generated*. Every downstream artifact inherits the
quality of this list.

In [ ]:
TICKET = """Add `is_valid_slug(text)` to text_utils.py.

A valid slug:
- is a str of 1-64 characters
- contains only lowercase letters, digits, and hyphens
- starts and ends with a letter or digit
- has no consecutive hyphens

Return True/False; never raise on bad input."""

draft = chat("Draft five acceptance criteria for this ticket, each one objectively "
             "testable:\n" + TICKET) if not course_ai.MOCK else course_ai.CANNED_91["acceptance"]
if course_ai.MOCK:
    print("(mock mode — canned draft; the live model drafts in class)\n")
print(draft)

In [ ]:
ACCEPTANCE = None
# YOUR CODE: edit the draft into the list you would sign off. Each item must be
# objectively testable — "handles edge cases well" is not a criterion.
if ACCEPTANCE is None:
    ACCEPTANCE = [
        "Accepts 1-64 chars of [a-z0-9-]; anything else returns False.",
        "Returns False for empty strings, non-str input, and length > 64.",
        "Returns False when the text starts or ends with a hyphen.",
        "Returns False for consecutive hyphens.",
        "Returns a bool for every input and never raises.",
    ]
    print("(canned criteria applied — edit the draft into your own above)\n")
assert len(ACCEPTANCE) >= 4, "fewer than four criteria is not a spec"
print("signed-off acceptance criteria:")
print("\n".join(f"  {i}. {c}" for i, c in enumerate(ACCEPTANCE, 1)))

### Step 2 — Tests FIRST, and the red bar (10 min)

Generate the pytest suite from the acceptance criteria — before any
implementation exists — and watch it fail. The red bar is not ceremony: it
proves the suite *can* fail, so green later means something. (Lab 5.1's loop,
compressed.)

In [ ]:
reply = chat("You are my AI pair. Write a pytest suite for the ticket and acceptance "
             "criteria below, as one python code block. The module is text_utils; the "
             "function does not exist yet.\n\nTicket:\n" + TICKET +
             "\n\nAcceptance criteria:\n" + "\n".join(ACCEPTANCE)) \
        if not course_ai.MOCK else course_ai.CANNED_91["tests"]
if course_ai.MOCK:
    print("(mock mode — canned test suite; the live model writes it in class)\n")

m = re.search(r"```(?:python)?\n(.*?)```", reply, re.DOTALL)
test_src = (m.group(1) if m else reply).strip() + "\n"
(WORKSPACE / "test_text_utils.py").write_text(test_src)
print(test_src)

(WORKSPACE / "text_utils.py").write_text(TEXT_UTILS_V0)   # pre-ticket module, no is_valid_slug
green, out = run_tests()
history.append(("tests before implementation (must be red)", green))
print(tail(out))
print("\nRED as expected?", not green)
assert not green, "the suite passed with no implementation — it has no teeth; fix it"

### Step 3 — Implementation, iterate to green (12 min)

Generate the implementation, run the suite, and if it is red, feed the failure
output back — the AI-TDD repair loop: the tests carry your intent, the model
carries the typing. The loop stops at three attempts. (Mock mode's canned
implementation is known-good, so the loop usually exits on the first pass; read
the loop anyway — with a key, the first attempt is often red.)

In [ ]:
reply = chat("You are my AI pair. Return the complete text_utils.py (keep the existing "
             "initials function and add is_valid_slug) implementing every acceptance "
             "criterion, as one python code block:\n" + "\n".join(ACCEPTANCE)) \
        if not course_ai.MOCK else course_ai.CANNED_91["module_v1"]

impl = WORKSPACE / "text_utils.py"
for attempt in range(1, 4):
    m = re.search(r"```(?:python)?\n(.*?)```", reply, re.DOTALL)
    impl.write_text((m.group(1) if m else reply).strip() + "\n")
    green, out = run_tests()
    history.append((f"implementation attempt {attempt}", green))
    print(f"--- attempt {attempt}: {'GREEN' if green else 'RED'} ---")
    if green:
        break
    reply = chat("The pytest run failed:\n" + tail(out) +
                 "\n\nReturn the complete corrected text_utils.py as one code block.") \
            if not course_ai.MOCK else course_ai.CANNED_91["module_v1"]

assert green, "still red after 3 attempts — tighten the criteria or repair by hand"
print("\nred -> green history:")
for label, ok in history:
    print(f"  [{'GREEN' if ok else 'RED  '}] {label}")

### Step 4 — The eval gate (10 min)

Tests prove the function works on the cases you thought of. The gate adds two
more layers, straight from Lab 6.1: a **golden set** (three cases chosen to
hurt — a valid multi-hyphen slug, mixed case with a space, consecutive hyphens)
and an **LLM-as-judge** rubric on the implementation itself. Both must pass to
ship.

In [ ]:
sys.path.insert(0, str(WORKSPACE))
import text_utils
importlib.reload(text_utils)   # in case the kernel cached the pre-ticket module

GOLDEN = [("release-notes-2026", True), ("Release Notes", False), ("a--b", False)]
prog_pass = 0
for s, expected in GOLDEN:
    got = text_utils.is_valid_slug(s)
    prog_pass += got == expected
    print(f"  is_valid_slug({s!r}) -> {got} (expected {expected}) {'ok' if got == expected else 'MISS'}")
print(f"programmatic: {prog_pass}/{len(GOLDEN)} golden cases")

JUDGE_SCHEMA = {"type": "object",
                "properties": {"score": {"type": "integer"}, "rationale": {"type": "string"}},
                "required": ["score", "rationale"]}

def score_judge(code):
    """Lab 6.1's judge idiom: rubric + structured 1-5 score. Mock mode uses a
    deterministic stand-in that scores the code's actual features."""
    if course_ai.MOCK:
        feats = ['"""' in code, "isinstance" in code, "64" in code, "fullmatch" in code]
        return (1 + sum(feats),
                "[MOCK] deterministic judge: docstring, type guard, length cap, single pattern")
    prompt = ("Rate the implementation 1-5 for correctness, clarity, and contract fit "
              "(5 = ship as-is). Answer as JSON with keys score (integer 1-5) and "
              "rationale (one sentence).\n\nAcceptance criteria:\n" +
              "\n".join(ACCEPTANCE) + "\n\nIMPLEMENTATION UNDER REVIEW:\n" + code)
    out = chat_json(prompt, schema=JUDGE_SCHEMA)
    return int(out["score"]), out["rationale"]

judge, why = score_judge(impl.read_text())
print(f"judge: {judge}/5 — {why}")

assert prog_pass == len(GOLDEN), "golden set failed — do not ship"
assert judge >= 4, "judge blocked the ship — read the rationale"
print("\nEVAL GATE PASSED — golden set 3/3 and judge >= 4. The ticket may ship.")

### Step 5 — The docs blurb (5 min)

Ship the artifact with words. Generate the README blurb, then check it names the
real constraints — a blurb that says "validates slugs" without the rules is
marketing, not documentation.

In [ ]:
blurb = chat("Write one README paragraph documenting is_valid_slug(text) for a user "
             "of the library: what it accepts, what it rejects, what it returns. "
             "No code block, just the paragraph.") \
        if not course_ai.MOCK else course_ai.CANNED_91["blurb"]
if course_ai.MOCK:
    print("(mock mode — canned blurb)\n")
print(blurb)

low = blurb.lower()
blurb_check = {"names the function": "is_valid_slug" in blurb,
               "states a constraint": ("hyphen" in low or "64" in low),
               "states the return": ("true" in low or "bool" in low)}
print("\nblurb checks:", {k: ("ok" if v else "MISSING") for k, v in blurb_check.items()})
assert all(blurb_check.values()), "blurb is missing substance — tighten the prompt and regenerate"

### Step 6 — The process scorecard (5 min)

The last artifact is about *you*, not the code: what did you verify with your
own eyes or a mechanical check — and what did you take on trust? Next ticket,
this card tells you where the remaining risk lives.

In [ ]:
process = {
    "acceptance criteria": {"verified": "edited the draft by hand against the ticket",
                            "trusted": "that five criteria cover the important edges"},
    "tests":               {"verified": "watched the suite go red before implementing",
                            "trusted": "that the generated tests encode the criteria faithfully"},
    "implementation":      {"verified": "suite green + eval gate (golden 3/3, judge >= 4)",
                            "trusted": "correctness on inputs outside the golden set"},
    "docs":                {"verified": "blurb names the real constraints (mechanical check)",
                            "trusted": "tone, formatting, completeness"},
}
# YOUR CODE: adjust to what you ACTUALLY verified this run — especially in live mode.

w = max(len(k) for k in process)
for stage, row in process.items():
    print(f"{stage:{w}}  verified: {row['verified']}")
    print(f"{'':{w}}  trusted : {row['trusted']}")

## Deliverable

1. Your edited acceptance criteria.
2. The red → green history and the passing eval gate (golden 3/3, judge ≥ 4).
3. The README blurb that passed all three checks.
4. Your process scorecard — plus one line on what you will verify *first* on the
   next ticket.

## Reflection

1. Which stage earned the model the most trust? The least? Why?
2. What would you add to the golden set before calling this ticket *done*-done?
3. Where in this run would a skipped verification have shipped a bug?

## Debrief (instructor-led)

1. Compare process scorecards: what did different people leave on trust — and is
   anyone's "verified" actually a "trusted"?
2. Three golden cases caught what they caught. How many is enough, and who
   decides?
3. Which of these six stages belong in CI, and which stay human forever?

## Troubleshooting

- **`ModuleNotFoundError: text_utils` in Step 4** — re-run Step 3 first (the
  module must exist), and keep the `sys.path.insert` / `importlib.reload` lines.
- **The red bar in Step 2 does not fire** — the suite has no teeth: check that
  the test file really imports `is_valid_slug`.
- **Green on the first attempt in mock mode** — expected: the canned
  implementation is known-good. Live runs often need the repair loop; the
  history table shows the mechanics either way.
- **Judge below 4 on a live run** — read the rationale; tighten the criteria or
  the implementation. Never weaken the rubric mid-flight to force a pass.
- **`No module named pytest`** — `pip install pytest` (pre-installed on the VM),
  into the same environment as the kernel.
- **`ModuleNotFoundError: course_ai`** — restart the kernel from the `labs/`
  folder and Run All.